# TEG — Verificación Tensorial del Apéndice K
**Miguel Ángel Franco León — Mayo 2026**

Verifica algebraicamente los pasos K.1–K.4 del Apéndice K:

1. `dim(H_int^(4)) = 2` via Clebsch–Gordan (SU(2))
2. `S_vN = ln 2` del estado máximamente mezclado
3. Teorema de equipartición: `σ_eff = σ_UV / 3`
4. Condición de cierre: exactamente 2 orientaciones globales
5. `b_eff = 8` → `d_s = ln 8 = D_eff`
6. Predicción de `r_J` (conjetura Apéndice L)

> **Sin funciones fantasma. Todo ejecutable celda a celda en Colab.**
> Ejecutar en orden, de arriba hacia abajo.

In [ ]:
# ── Celda 1: Imports ──────────────────────────────────────
# numpy y scipy vienen preinstalados en Colab
import numpy as np
from collections import Counter
import itertools

print('Imports OK')
print('Python y numpy listos')

---
## Cadena de derivación completa

Desde el axioma tetraédrico `z_fund = 4` hasta `σ_eff = 0.1088`.

In [ ]:
# ── Celda 2: Constantes TEG ───────────────────────────────
z_fund    = 4
z_pack    = 12
D_A       = np.log(z_fund)                                      # ln 4
D_V       = np.log(2 * z_fund)                                  # ln 8
holo_bit  = D_V - D_A                                           # ln 2 exacto
sigma_UV  = 2 * np.log(z_pack / (z_pack - z_fund)) / np.log(z_pack)
N_bits    = D_V / np.log(2)                                     # 3 exacto
sigma_eff = sigma_UV / N_bits
partial   = 3 - D_V                                             # codimension

print('Constantes TEG derivadas:')
print(f'  D_A  = ln(4)           = {D_A:.10f}')
print(f'  D_V  = ln(8)           = {D_V:.10f}')
print(f'  Bit holografico = ln2  = {holo_bit:.10f}')
print(f'  sigma_UV               = {sigma_UV:.10f}')
print(f'  N_bits  (debe ser 3)   = {N_bits:.10f}')
print(f'  sigma_eff              = {sigma_eff:.10f}')
print(f'  partial = 3 - ln8      = {partial:.10f}')

assert abs(holo_bit - np.log(2)) < 1e-12, 'FALLO: bit holografico'
assert abs(N_bits   - 3.0)       < 1e-12, 'FALLO: N_bits'
print()
print('OK: bit holografico = ln2  (identidad algebraica exacta)')
print('OK: N_bits = 3             (entero exacto = dim(R3))')

---
## Paso K.1 — `dim(H_int^(4)) = 2` via Clebsch–Gordan

Construimos el Casimir total $J^2$ para 4 espines $j=1/2$
y contamos los autovalores con $J=0$.

$$\left(\tfrac{1}{2}\otimes\tfrac{1}{2}\right)\otimes\left(\tfrac{1}{2}\otimes\tfrac{1}{2}\right) = 0\oplus 0\oplus 1\oplus 1\oplus 2$$

El subespacio $J=0$ tiene dimensión **exactamente 2**.

In [ ]:
# ── Celda 3: Operadores SU(2) y Casimir J^2 ───────────────
J_z     = np.array([[0.5, 0], [0, -0.5]], dtype=complex)
J_plus  = np.array([[0, 1], [0, 0]],      dtype=complex)
J_minus = np.array([[0, 0], [1, 0]],      dtype=complex)
I       = np.eye(2, dtype=complex)

def kron4(A, B, C, D):
    return np.kron(np.kron(np.kron(A, B), C), D)

Jz_tot    = (kron4(J_z,I,I,I)    + kron4(I,J_z,I,I) +
             kron4(I,I,J_z,I)    + kron4(I,I,I,J_z))
Jplus_tot = (kron4(J_plus,I,I,I) + kron4(I,J_plus,I,I) +
             kron4(I,I,J_plus,I) + kron4(I,I,I,J_plus))
Jm_tot    = Jplus_tot.conj().T
J2_tot    = (Jz_tot @ Jz_tot +
             0.5 * (Jplus_tot @ Jm_tot + Jm_tot @ Jplus_tot))

print('Operadores SU(2) construidos (espacio 16x16)')
print(f'Forma de J2_tot: {J2_tot.shape}')

In [ ]:
# ── Celda 4: Autovalores de J^2 ────────────────────────────
eigs   = np.round(np.real(np.linalg.eigvalsh(J2_tot)), 8)
counts = Counter(eigs)
dim_J0 = counts.get(0.0, 0)

print('Descomposicion de Clebsch-Gordan:')
for ev, mult in sorted(counts.items()):
    j_val = (-1 + np.sqrt(max(0.0, 1 + 4*ev))) / 2
    print(f'  j(j+1) = {ev:.4f}  ->  j = {j_val:.1f}   multiplicidad = {mult}')

print()
print(f'dim(subespacio J=0) = {dim_J0}  (debe ser 2)')
assert dim_J0 == 2, 'FALLO: dim != 2'
print('OK: dim(H_int^(4)) = 2  VERIFICADO')

---
## Paso K.2 — Base explícita de intertwiners

Construimos $|v_+\rangle$ y $|v_-\rangle$ **directamente en el espacio 16D**
(4 espines, $2^4 = 16$ dimensiones).

> **Nota técnica:** El error frecuente es usar `np.kron` entre dos vectores de
> dimensión 16, lo que produce un vector de dimensión 256 (8 espines).
> La construcción correcta usa `ket()` que vive en 16D directamente.

In [ ]:
# ── Celda 5: Base de intertwiners (corregida) ──────────────
# CORRECTO: ket() vive en el espacio 16D de 4 espines
# 0 = up, 1 = down; indice = s1*8 + s2*4 + s3*2 + s4
def ket(s1, s2, s3, s4):
    v = np.zeros(16, dtype=complex)
    v[s1*8 + s2*4 + s3*2 + s4] = 1.0
    return v

# |v+> acoplamiento (12)(34)
v_plus = 0.5 * (
    ket(0,1,0,1) - ket(0,1,1,0)
    - ket(1,0,0,1) + ket(1,0,1,0)
)

# |v-> acoplamiento (13)(24)
v_minus = 0.5 * (
    ket(0,0,1,1) - ket(0,1,1,0)
    - ket(1,0,0,1) + ket(1,1,0,0)
)

# Verificar que son estados J=0
ev_p = np.real(v_plus.conj()  @ (J2_tot @ v_plus))
ev_m = np.real(v_minus.conj() @ (J2_tot @ v_minus))
print(f'<v+|J2|v+> = {ev_p:.2e}  (debe ser 0)')
print(f'<v-|J2|v-> = {ev_m:.2e}  (debe ser 0)')
assert ev_p < 1e-12 and ev_m < 1e-12, 'FALLO: no son estados J=0'

# Gram-Schmidt
v1     = v_plus / np.linalg.norm(v_plus)
v2_raw = v_minus - (v1.conj() @ v_minus) * v1
norm2  = np.linalg.norm(v2_raw)
print(f'Norma de v2 antes de normalizar: {norm2:.6f}  (debe ser > 0)')
assert norm2 > 0.1, 'FALLO: v_plus y v_minus son linealmente dependientes'
v2      = v2_raw / norm2
overlap = abs(v1.conj() @ v2)
print(f'Solapamiento tras Gram-Schmidt: {overlap:.2e}  (debe ser ~0)')
assert overlap < 1e-12
print('OK: base ortonormal {|v+>, |v->} VERIFICADA')

---
## Paso K.3 — Entropía de Von Neumann = ln 2

El estado de máxima entropía en $\mathcal{H}_{int}^{(4)}$ (dimensión 2) es
$\rho_{max} = I_2/2$, con entropía exactamente $\ln 2$.

Esto coincide con el **bit holográfico** $D_V - D_A = \ln 2$.

In [ ]:
# ── Celda 6: Entropia de Von Neumann ───────────────────────
lam  = np.array([0.5, 0.5])
S_vN = -np.sum(lam * np.log(lam))
diff = abs(S_vN - np.log(2))

print('Estado maximalmente mezclado en H_int^(4): rho_max = I2/2')
print(f'S_vN(rho_max)  = {S_vN:.15f}')
print(f'ln(2)          = {np.log(2):.15f}')
print(f'Diferencia     = {diff:.2e}')
assert diff < 1e-12, 'FALLO: S_vN != ln2'
print('OK: S_vN = ln2  VERIFICADO')
print()
print(f'Bit holografico D_V - D_A = {holo_bit:.15f}')
print(f'Son identicos: {abs(S_vN - holo_bit) < 1e-12}')

---
## Teorema de Equipartición — `σ_eff = σ_UV / 3`

El estado $\rho_{total} = I_8/8$ es el único estado en $(\mathbb{C}^2)^{\otimes 3}$
invariante bajo $S_3$ con entropía máxima.
La traza parcial da $S(\rho_x)/S(\rho_{total}) = \ln 2 / \ln 8 = 1/3$.

In [ ]:
# ── Celda 7: Simetria S3 e invarianza ──────────────────────
def perm_matrix(perm, n=3):
    dim = 2**n
    P   = np.zeros((dim, dim), dtype=complex)
    for i in range(dim):
        bits     = [(i >> (n-1-k)) & 1 for k in range(n)]
        new_bits = [bits[perm[k]] for k in range(n)]
        j        = sum(new_bits[k] * (2**(n-1-k)) for k in range(n))
        P[j, i]  = 1.0
    return P

rho_total = np.eye(8, dtype=complex) / 8.0
P12       = perm_matrix([1, 0, 2])
P23       = perm_matrix([0, 2, 1])
c12       = np.max(np.abs(rho_total @ P12 - P12 @ rho_total))
c23       = np.max(np.abs(rho_total @ P23 - P23 @ rho_total))

print(f'|[rho_total, P12]|_max = {c12:.2e}  (debe ser ~0)')
print(f'|[rho_total, P23]|_max = {c23:.2e}  (debe ser ~0)')
assert c12 < 1e-14 and c23 < 1e-14
print('OK: rho_total = I8/8 es S3-invariante')

In [ ]:
# ── Celda 8: Equiparticion y sigma_eff ─────────────────────
fraction     = np.log(2) / np.log(8)    # = 1/3 exacto
sigma_eff_th = sigma_UV * fraction
sparc_val    = 0.108
agreement    = abs(sigma_eff_th - sparc_val) / sparc_val * 100

print(f'S(rho_x) / S(rho_total) = ln2/ln8 = 1/3 = {fraction:.10f}')
print(f'Diferencia de 1/3: {abs(fraction - 1/3):.2e}  (exacto)')
print()
print(f'sigma_UV              = {sigma_UV:.6f}')
print(f'sigma_eff (derivado)  = {sigma_eff_th:.6f}')
print(f'sigma_eff (SPARC)     = {sparc_val:.6f}')
print(f'Acuerdo               = {agreement:.2f}%')
assert abs(fraction - 1/3) < 1e-14
assert agreement < 1.0
print('OK: sigma_eff = sigma_UV / 3  VERIFICADO')
print('OK: acuerdo con SPARC < 1%')

---
## Condición de cierre — 2 orientaciones globales

La condición $\sum_{i=1}^4 s_i \hat{n}_i = 0$ con $s_i = \pm 1$
tiene exactamente **2 soluciones**, correspondientes a las
dos quiralidades $N_+$ y $N_-$ del 4-simplex Lorentziano.
Esto garantiza $K_{node} = 2$ sin interferencia destructiva.

In [ ]:
# ── Celda 9: Condicion de cierre del tetraedro ─────────────
sq3     = np.sqrt(3)
verts   = np.array([
    [ 1,  1,  1],
    [ 1, -1, -1],
    [-1,  1, -1],
    [-1, -1,  1]
], dtype=float) / sq3
normals = -verts   # normales hacia afuera

print('Normales a las 4 caras del tetraedro regular:')
for i, n in enumerate(normals):
    print(f'  n{i+1} = [{n[0]:+.4f}, {n[1]:+.4f}, {n[2]:+.4f}]')

solutions = []
for signs in itertools.product([-1, 1], repeat=4):
    s       = np.array(signs)
    closure = np.dot(s, normals)
    if np.linalg.norm(closure) < 1e-10:
        solutions.append(signs)

print()
print('Soluciones a sum(si * ni) = 0:')
for sol in solutions:
    print(f'  (s1,s2,s3,s4) = {sol}')
print()
print(f'Numero de soluciones = {len(solutions)}  (debe ser 2)')
assert len(solutions) == 2, 'FALLO: numero de soluciones != 2'
s0 = np.array(solutions[0])
s1 = np.array(solutions[1])
antipodal = np.allclose(s0 + s1, 0)
print(f'Son antipodas (s -> -s): {antipodal}')
print('OK: exactamente 2 soluciones (quiralidades N+ y N-)  VERIFICADO')
print('OK: K_node = 2 sin interferencia destructiva')

---
## Paso K.4 — `b_eff = 8` y dimensión espectral `d_s = ln 8`

$$b_{eff} = K_{node} \times z_{fund} = 2 \times 4 = 8$$
$$d_s = \ln b_{eff} = \ln 8 = D_{eff}$$

In [ ]:
# ── Celda 10: Factor de ramificacion y dimension espectral ─
K_node = 2        # dim(H_int^(4)) verificado en Celda 4
b_eff  = K_node * z_fund
d_s    = np.log(b_eff)
in_lqg = (2.0 <= d_s <= 2.2)
in_cdt = (2.0 <= d_s <= 2.5)
lqg_ok = 'OK' if in_lqg else 'FALLO'
cdt_ok = 'OK' if in_cdt else 'FALLO'

print(f'K_node = dim(H_int^(4))  = {K_node}')
print(f'z_fund                   = {z_fund}')
print(f'b_eff  = K_node x z_fund = {b_eff}')
print(f'd_s    = ln(b_eff)        = {d_s:.10f}')
print(f'D_eff  = ln(8)            = {D_V:.10f}')
print(f'd_s == D_eff: {abs(d_s - D_V) < 1e-14}')
print()
print(f'd_s en rango LQG [2.0, 2.2]: {in_lqg}  [{lqg_ok}]')
print(f'd_s en rango CDT [2.0, 2.5]: {in_cdt}  [{cdt_ok}]')
assert b_eff == 8
assert abs(d_s - D_V) < 1e-14
assert in_lqg and in_cdt
print('OK: b_eff=8, d_s=ln8=D_eff  VERIFICADO')

---
## Apéndice L — Predicción de `r_J` (conjetura UV–IR)

$$r_J = \frac{\ell_{Pl}^{\sigma_{eff}} \cdot R_H^{1-\sigma_{eff}}}{\partial \times \sqrt{\pi}}$$

Todos los factores son derivados del axioma tetraédrico.
$R_H = c/H_0$ es la escala IR externa (Open Problem 5).

In [ ]:
# ── Celda 11: Prediccion de r_J ────────────────────────────
l_Pl = 1.616e-35   # m
kpc  = 3.086e19    # m por kpc

print(f'Factor geometrico: partial x sqrt(pi) = {partial * np.sqrt(np.pi):.6f}')
print()
print(f'{"H0 (km/s/Mpc)":>18}  {"Fuente":>10}  {"r_J (kpc)":>10}  '
      f'{"vs 0.62 ref":>12}  {"vs 0.58 SPARC":>14}')
print('-' * 72)

for H0_kms, label in [(67.4,'Planck'), (70.0,'CCHP'), (73.0,'SH0ES')]:
    H0      = H0_kms * 1e3 / 3.086e22
    R_H     = 2.998e8 / H0
    r_J     = (l_Pl**sigma_eff * R_H**(1 - sigma_eff)) / (partial * np.sqrt(np.pi))
    r_J_kpc = r_J / kpc
    e1      = (r_J_kpc - 0.62) / 0.62 * 100
    e2      = (r_J_kpc - 0.58) / 0.58 * 100
    print(f'{H0_kms:18.1f}  {label:>10}  {r_J_kpc:10.4f}  {e1:+10.1f}%  {e2:+12.1f}%')

print()
print('Nota: conjetura UV-IR del Apendice L.')
print('Requiere verificacion via TNR del Apendice K (Open Problem 7).')

---
## Resumen final

In [ ]:
# ── Celda 12: Resumen completo ─────────────────────────────
print('=' * 60)
print('RESUMEN — APENDICE K')
print('TEG — Miguel Angel Franco Leon, 2026')
print('=' * 60)

resultados = [
    ('dim(H_int^(4)) = 2',      dim_J0 == 2),
    ('|v+> y |v-> son J=0',     ev_p < 1e-12 and ev_m < 1e-12),
    ('Base ortonormal OK',       overlap < 1e-12),
    ('S_vN = ln2',              abs(S_vN - np.log(2)) < 1e-12),
    ('Bit holografico = ln2',   abs(holo_bit - np.log(2)) < 1e-12),
    ('N_bits = 3 exacto',       abs(N_bits - 3) < 1e-12),
    ('rho_total S3-invariante',  c12 < 1e-14 and c23 < 1e-14),
    ('Fraccion 1/3 exacta',      abs(fraction - 1/3) < 1e-14),
    ('sigma_eff = sigma_UV/3',  abs(sigma_eff - sigma_UV/3) < 1e-12),
    ('Acuerdo SPARC < 1pct',    agreement < 1.0),
    ('Soluciones cierre = 2',   len(solutions) == 2),
    ('b_eff = 8',               b_eff == 8),
    ('d_s = ln8 = D_eff',       abs(d_s - D_V) < 1e-14),
    ('d_s en rango LQG',        in_lqg),
    ('d_s en rango CDT',        in_cdt),
]

all_pass = True
for desc, ok in resultados:
    estado = 'OK   ' if ok else 'FALLO'
    print(f'  [{estado}]  {desc}')
    if not ok:
        all_pass = False

print()
if all_pass:
    print('  TODOS LOS PASOS K.1-K.4 VERIFICADOS')
    print('  El nucleo tensorial del Apendice K es solido.')
    print()
    print('  Pendiente (Open Problem 7):')
    print('  Verificar que K_node=2 se preserva bajo')
    print('  coarse-graining iterativo via TNR/SL2Cfoam.')
else:
    print('  ALGUN PASO FALLO — revisar celdas anteriores')